# 11: tabela główna: wszystkie metody na tym samym val, tą samą miarą

Bez treningu: ocena istniejących runów i metod bez uczenia na val `rs_video_x4k` (321 próbek, 13 scen spoza treningu).
Kod: `src/main_table.py` (opis metod i miar w nagłówku pliku).

* **`psnr_full`, `ssim_full`** = cały obraz, średnia po obrazach: **wspólna miara, kolumna główna**. Wcześniejsze tabele
  mieszały ją z `psnr` liczonym na obszarze widocznym w A (`train_raft.py` liczył tylko ten drugi); tu `psnr_vis` jest tylko
  dla porównania ze starymi liczbami.
* **CI95 (sceny)**: przedział ufności średniej z bootstrapu **po scenach** (val ma 13 scen po 27 próbek, próbki z jednej
  sceny nie są niezależne). Przedziały wartości bezwzględnych są szerokie; o przewadze jednej metody nad drugą mówi
  **różnica sparowana** (te same próbki) względem modelu głównego, z własnym CI i liczbą próbek, w których metoda wygrywa.
* **ms/parę**: mediana 20 przebiegów na jednej parze po rozgrzewce, sama metoda (bez wczytywania danych), na podanym
  urządzeniu; metody klasyczne na CPU (OpenCV + geometria w PyTorch).
* **kroki treningu**: epoki × (próbki / bs) danego runu (bez wcześniejszych etapów: RAFT jest pretrenowany poza projektem,
  `refine_ft` startuje z `refine`).

Nowe punkty odniesienia: **samo B, średnia A+B** (trywialne) i **klasyczny przepływ (DIS, Farnebäck z OpenCV) + nasza
geometria**: bez żadnego uczenia, bez pretreningu. Sprawdzone lokalnie 14.11: DIS + geometria v1 33,44 dB psnr_full,
czyli więcej niż każda sieć bez geometrii.

**Środowisko:** GPU (L4 albo T4) dla RAFT i sieci; całość ~15–25 min, w tym metody klasyczne na CPU ~6 min.
Wyniki zapisują się na bieżąco (`results/tabela_glowna/per_sample/<metoda>.csv`), po rozłączeniu uruchom znowu: policzone
metody są pomijane (`REDO = True` liczy od nowa).

In [16]:
# === SETUP: uruchamiaj zawsze jako pierwszą komórkę ===
# Kod jest pobierany z GitHuba na maszynę Colaba (lokalne pliki z VS Code tam nie trafiają).
# Po zmianie kodu: commit + push w VS Code, potem ponownie ta komórka (robi git pull).
REPO = 'kordiusz/RollingShutter'   # repozytorium na GitHubie
BRANCH = 'main'

import os, sys, subprocess, getpass, importlib
from google.colab import drive
drive.mount('/content/drive')

TOKEN_FILE = '/content/drive/MyDrive/rs_project/.github_token'   # opcjonalnie: token zapisany na Drive
CODE = '/content/code'

def _git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(token, '***') if 'token' in globals() else r.stderr
        raise RuntimeError('git ' + args[0] + ' nie powiodło się:\n' + msg)
    return r.stdout

if not os.path.exists(CODE):
    token = (open(TOKEN_FILE).read().strip() if os.path.exists(TOKEN_FILE)
             else getpass.getpass('GitHub token (fine-grained, tylko odczyt): '))
    _git('clone', '-q', '-b', BRANCH, f'https://{token}@github.com/{REPO}.git', CODE)
else:
    _git('-C', CODE, 'pull', '-q')
os.chdir(CODE)
sys.path.insert(0, f'{CODE}/src')
import project_paths as P
importlib.reload(P)
P.ensure_dirs()
print(_git('log', '-1', '--format=kod: %h  %s  (%cd)', '--date=short'))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
kod: 3f646f6  Notebook 09 with results (camera-B ablation run on Colab)  (2026-11-14)

NVIDIA L4, 23034 MiB


## 1. Zbiór danych (rozpakowanie z Drive na dysk maszyny Colaba)

In [17]:
import os, glob, shutil, subprocess
DATASET_NAME = 'rs_video_x4k'
z = P.find_dataset_zip(DATASET_NAME)
assert z, f'Brak {DATASET_NAME}.zip w {P.DATASETS} - wrzuć tam spakowany folder data/{DATASET_NAME}'
UNZ = f'/content/unz_{DATASET_NAME}'
DONE = f'{UNZ}/.rozpakowane'          # znacznik: rozpakowanie skończyło się bez błędu
if not os.path.exists(DONE):
    shutil.rmtree(UNZ, ignore_errors=True)            # resztki po przerwanym rozpakowaniu
    LOCAL_ZIP = f'/content/{DATASET_NAME}.zip'
    # najpierw kopia na dysk Colaba: unzip czytający wprost z Drive potrafi się urwać ("zipfile read error")
    print(f'kopiuję {z} ({os.path.getsize(z):,} B) na dysk Colaba ...')
    subprocess.run(['cp', str(z), LOCAL_ZIP], check=True)
    t = subprocess.run(['unzip', '-tq', LOCAL_ZIP], capture_output=True, text=True)
    assert t.returncode == 0, ('Archiwum uszkodzone - wgraj zip na Drive ponownie (lokalnie ma 1 233 434 992 B):\n'
                               + (t.stdout + t.stderr)[-1500:])
    print('archiwum OK, rozpakowuję ...')
    subprocess.run(['unzip', '-q', LOCAL_ZIP, '-d', UNZ], check=True)
    os.remove(LOCAL_ZIP)
    open(DONE, 'w').close()
# folder z podfolderami train/ i val/ (niezależnie od tego, jak spakowano)
DATA = next(os.path.dirname(p) for p in sorted(glob.glob(f'{UNZ}/**/train', recursive=True)))
assert os.path.isdir(f'{DATA}/val'), f'brak {DATA}/val - usuń {UNZ} i uruchom komórkę ponownie'
print('DATA =', DATA, '|', len(os.listdir(f'{DATA}/train')), 'train /', len(os.listdir(f'{DATA}/val')), 'val')
WORKERS = min(8, os.cpu_count())

DATA = /content/unz_rs_video_x4k/rs_video_x4k | 2397 train / 321 val


## 2. Ocena wszystkich metod

In [18]:
import main_table as mt
REDO = False
OUT = P.RESULTS / 'tabela_glowna'
TAB = mt.run(DATA, OUT, runs=P.RUNS, redo=REDO, workers=WORKERS)

input: psnr_full 26.35 dB (1 s)
only_b: psnr_full 19.50 dB (1 s)
mean_ab: psnr_full 23.47 dB (1 s)
farneback_v1: psnr_full 31.84 dB (67 s)
dis_v1: psnr_full 33.44 dB (61 s)
dis_v2: psnr_full 33.45 dB (138 s)
Downloading: "https://download.pytorch.org/models/raft_large_C_T_SKHT_V2-ff5fadd5.pth" to /root/.cache/torch/hub/checkpoints/raft_large_C_T_SKHT_V2-ff5fadd5.pth


100%|██████████| 20.1M/20.1M [00:00<00:00, 82.1MB/s]


raft_zs_v1: psnr_full 32.34 dB (6 s)
raft_zs_v2: psnr_full 32.81 dB (11 s)
raft_ft_v1: psnr_full 33.60 dB (6 s)
raft_ft2_v2: psnr_full 33.89 dB (11 s)
ae: psnr_full 23.27 dB (2 s)
unet_plain30: psnr_full 28.42 dB (2 s)
unet_plain200: psnr_full 30.94 dB (2 s)
corr80: psnr_full 31.06 dB (2 s)
unet_plain_single: psnr_full 26.51 dB (2 s)
refine: psnr_full 35.24 dB (7 s)
refine_ft: psnr_full 35.37 dB (7 s)


## 3. Tabela
Kolumna `różnica vs refine_ft` = średnia różnica psnr_full (dB) względem modelu głównego na tych samych próbkach;
`CI95 różnicy` nie zawiera 0 = różnica pewna mimo małej liczby scen.

In [19]:
import pandas as pd
with pd.option_context('display.max_columns', 30, 'display.width', 250):
    display(TAB.drop(columns=['klucz']).set_index(['grupa', 'metoda']).round(3))
print('zapisano', OUT / 'tabela_glowna.csv')

psnr_full    CI95 (sceny)  mediana  ssim_full  psnr_vis  różnica vs refine_ft      CI95 różnicy lepsze od refine_ft  ms/parę    urządzenie param. [M]  \
grupa                          metoda                                                                                                                                                                                                 
trywialne                      bez korekcji (obraz A)                            26.348  [23.51; 28.96]   26.086      0.797    26.348                -9.020   [-10.84; -7.32]               8/321      0.0     NVIDIA L4              
                               sam obraz B                                       19.499  [18.59; 20.52]   18.687      0.575    19.499               -15.869  [-17.00; -14.24]               0/321      0.0     NVIDIA L4              
                               średnia A i B                                     23.469  [22.06; 24.85]   23.046      0.740    23.469               -11.899  [-12.79; -10.57]               0/321      0.0     NVIDIA L4              
przepływ klasyczny + geometria Farnebäck + geometria v1                          31.837  [29.72; 33.70]   32.484      0.910    32.280                -3.531    [-4.29; -2.82]              17/321    194.2  CPU (OpenCV)              
                               DIS + geometria v1                                33.443  [31.23; 35.33]   34.020      0.935    34.104                -1.925    [-2.40; -1.52]              41/321    179.0  CPU (OpenCV)              
                               DIS + geometria v2                                33.445  [31.31; 35.31]   33.992      0.935    34.028                -1.923    [-2.44; -1.50]              34/321    546.5  CPU (OpenCV)              
RAFT + geometria               RAFT gotowy + geometria v1                        32.336  [30.42; 34.05]   32.566      0.933    32.857                -3.032    [-3.59; -2.57]               4/321     48.0     NVIDIA L4       5.26   
                               RAFT gotowy + geometria v2                        32.812  [30.94; 34.44]   33.075      0.936    33.249                -2.556    [-2.97; -2.19]               7/321     59.1     NVIDIA L4       5.26   
                               RAFT dostrojony (04) + geometria v1               33.603  [31.58; 35.33]   34.207      0.938    34.300                -1.765    [-2.00; -1.56]              15/321     48.0     NVIDIA L4       5.26   
                               RAFT dostrojony z v2 (07) + geometria v2          33.887  [31.93; 35.54]   34.304      0.940    34.460                -1.481    [-1.68; -1.32]              17/321     59.7     NVIDIA L4       5.26   
sieci bez geometrii            autoenkoder bez skipów (ae)                       23.274  [22.20; 24.31]   23.079      0.681    23.274               -12.094  [-13.26; -10.51]               0/321      4.4     NVIDIA L4       10.2   
                               zwykły U-Net, 30 ep.                              28.416  [26.01; 30.52]   28.709      0.846    28.455                -6.952    [-8.36; -5.65]               9/321      4.3     NVIDIA L4       2.72   
                               zwykły U-Net, 200 ep.                             30.943  [28.40; 33.17]   31.759      0.895    31.045                -4.425    [-5.64; -3.43]              19/321      4.3     NVIDIA L4       2.72   
                               enkoder + korelacja (corr), 80 ep.                31.062  [28.63; 33.25]   31.492      0.887    31.062                -4.306    [-5.53; -3.22]              14/321     11.3     NVIDIA L4       6.24   
ablacja: bez kamery B          zwykły U-Net bez kamery B, 200 ep.                26.511  [23.79; 29.04]   26.419      0.810    26.529                -8.857   [-10.59; -7.24]               6/321      4.2     NVIDIA L4       2.72   
RAFT + geometria + U-Net       RAFT (zamrożony) + geometria + U-Net              35.241  [33.29; 36.91]   35.549      0.953    35.676                -0

zapisano /content/drive/MyDrive/rs_project/results/tabela_glowna/tabela_glowna.csv


## 4. Po ocenie
Skopiuj `tabela_glowna.csv` do `docs/` (albo daj znać, wpiszę do `experiments/README.md`). Odłącz środowisko.

## 5. Demo wizualne: sieci bez geometrii kontra geometria (`src/demo_models.py`)

Te same próbki, obok siebie: RS A | RS B | autoenkoder (`ae`) | zwykły U-Net 200 ep. | `corr` | RAFT + geometria (etap modelu
głównego, bez U-Netu) | model główny | GS. Pod każdym obrazem PSNR całego obrazu. Drugi rząd: powiększenie obszaru
z najsilniejszym RS (żółta ramka na RS A), trzeci rząd: błąd ×4 (czarne = idealnie). Do tego GIF dla każdej próbki (`modele_<próbka>.gif`, ten sam styl co demo w `docs/podsumowanie_2026-11-16/podsumowanie_badan.md`:
RS A → ae → zwykły U-Net → corr → model główny).
Próbki: przykłady z repo (`examples/`) i 3 próbki val z najsilniejszym RS (najniższy PSNR wejścia, różne sceny;
potrzebny wynik sekcji 2). Wystarczy CPU, na GPU kilka sekund. `corrgeom` pominięty: żaden run nie dał działających wag.

In [ ]:
import json, pandas as pd, subprocess
from pathlib import Path
DEMO = P.RESULTS / 'demo_modeli'
DEMO.mkdir(parents=True, exist_ok=True)
samples = ['examples/x4k_val_00276_02', 'examples/x4k_val_00273_02', 'examples/kolo']
inp = OUT / 'per_sample' / 'input.csv'
if inp.exists():                                   # 3 próbki val z najsilniejszym RS, każda z innej sceny
    df = pd.read_csv(inp).sort_values('psnr_full')
    df['scena'] = [Path(json.load(open(f'{DATA}/val/{x}/meta.json'))['source'].replace(chr(92), '/')).parts[0] for x in df['sample']]
    df = df[~df['sample'].isin(['00276_02', '00273_02'])].drop_duplicates('scena').head(3)
    samples += [f'{DATA}/val/{s}' for s in df['sample']]
    print(df[['sample', 'psnr_full']].to_string(index=False))
else:
    print('brak', inp, '- uruchom sekcję 2, na razie tylko przykłady z repo')
r = subprocess.run(['python', 'src/demo_models.py', *samples, '--runs', str(P.RUNS),
                    '--out', str(DEMO / 'porownanie_modeli.jpg'), '--gif-dir', str(DEMO / 'gify')],
                   capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
from IPython.display import Image as IPImage, display
display(IPImage(str(DEMO / 'porownanie_modeli.jpg'), width=1400))
print('zapisano w', DEMO, '- skopiuj gify/modele_x4k_val_00276_02.gif, gify/modele_x4k_val_00273_02.gif, gify/modele_kolo.gif'
      ' i porownanie_modeli.jpg do docs/podsumowanie_2026-11-16/demo_modeli/')